In [ ]:
import pandas as pd
import numpy as np
from ml_utils.src import *
from ml_utils.config import *

from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline

# 1.0 - Classificação de Presença Baseada em Fatores Socioeconômicos Usando SVM

In [ ]:
colunas = ['Q001','Q002','Q003', 'Q004', 'Q005', 'Q006', 'Q007', 'Q008', 'Q009', 'Q010', 'Q011', 'Q012', 'Q013', 'Q014', 'Q015', 'Q016', 'Q017', 'Q018', 
           'Q019', 'Q020', 'Q021', 'Q022', 'Q023', 'Q024', 'Q025', 'TP_PRESENCA_LC', 'TP_PRESENCA_CH', 'TP_PRESENCA_CN', 'TP_PRESENCA_MT', 
           'TP_FAIXA_ETARIA', 'TP_SEXO','TP_ESTADO_CIVIL', 'TP_COR_RACA', 'TP_ESCOLA', 'TP_ST_CONCLUSAO', 'IN_TREINEIRO', 
           'NU_ANO', 'TP_LOCALIZACAO_ESC','TP_SIT_FUNC_ESC', 'NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC', 'NU_NOTA_MT', 'NU_NOTA_REDACAO', 'TP_DEPENDENCIA_ADM_ESC']

df = pd.read_parquet(DATA_DIR, columns = colunas)

## 1.2- Pré-Processando os Dados

In [ ]:
df = preparar_dados(df, objetivo = 'Desempenho', n_samples = 50000)

## 1.3- Construção da Matriz X e Vetor y

In [4]:
X = df.drop(columns=['MEDIA', 'FALTOU'])
y_media = df['MEDIA']

## 1.4 - Separação em Dados de Treino, Validação e Teste

In [5]:
X_train, X_test, y_train, y_test = train_test_split(X, y_media, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [6]:
quantil = y_train.quantile(0.5)
y_train = (y_train >= quantil).astype(int)
y_val   = (y_val   >= quantil).astype(int)
y_test  = (y_test  >= quantil).astype(int)

y_train = y_train.map({0: -1, 1: 1})
y_val = y_val.map({0: -1, 1: 1})
y_test = y_test.map({0: -1, 1: 1})

## 1.5 - Tratando os Dados

In [ ]:
preprocessador = pre_processor(X_train)

X_train = preprocessador.transform(X_train).astype(np.float32)
X_val   = preprocessador.transform(X_val).astype(np.float32)
X_test  = preprocessador.transform(X_test).astype(np.float32)

## 1.6 - Treinando o SVM

In [12]:
pipe = Pipeline([
    ('svm', SVC(kernel='rbf', class_weight='balanced', random_state=42))
])

params = {
    'svm__C': [0.001, 0.01, 0.1, 1, 10],
    'svm__gamma': [0.001, 0.01, 0.1, 1]
}

grid = GridSearchCV(pipe, params, cv=5, scoring='f1', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)  

print(grid.best_params_)

Fitting 5 folds for each of 20 candidates, totalling 100 fits
{'svm__C': 1, 'svm__gamma': 0.1}


## 1.7 - Validando o SVM

In [13]:
print(classification_report(y_val, grid.predict(X_val)))

              precision    recall  f1-score   support

          -1       0.68      0.77      0.72       848
           1       0.75      0.65      0.70       904

    accuracy                           0.71      1752
   macro avg       0.71      0.71      0.71      1752
weighted avg       0.72      0.71      0.71      1752



## 1.8 Treinando o SVM com Todos os Dados

In [16]:
X_train, X_test, y_train, y_test = train_test_split(X, y_media, test_size=0.2, random_state=42)

quantil = y_train.quantile(0.5)
y_train = (y_train >= quantil).astype(int)
y_test  = (y_test  >= quantil).astype(int)

y_train = y_train.map({0: -1, 1: 1})
y_test = y_test.map({0: -1, 1: 1})


X_train = preprocessador.fit_transform(X_train).astype(np.float32)
X_test  = preprocessador.transform(X_test).astype(np.float32) 

svm = SVC(kernel='rbf', class_weight='balanced', random_state=42, C = 1, gamma= 0.1)

svm.fit(X_train, y_train)

print(classification_report(y_test, svm.predict(X_test)))

              precision    recall  f1-score   support

          -1       0.69      0.73      0.71      1136
           1       0.69      0.64      0.66      1054

    accuracy                           0.69      2190
   macro avg       0.69      0.69      0.69      2190
weighted avg       0.69      0.69      0.69      2190

